# SpecialistLLM — Medical Q&A Fine-Tuning with QLoRA

**Goal**: Fine-tune TinyLlama-1.1B to become a medical Q&A specialist using QLoRA.

**Run this on Google Colab** (free T4 GPU). Training takes ~20-30 minutes.

## What We'll Do
1. Load a pre-trained LLM (TinyLlama 1.1B)
2. Apply QLoRA (4-bit quantization + LoRA adapters)
3. Fine-tune on 5,000 medical Q&A pairs
4. Compare base model vs fine-tuned model
5. Save the adapter weights (~10MB)

## Key Concept
We're NOT training from scratch. We're adjusting ~1% of the model's parameters
to make it better at medical Q&A. The base model already knows English, reasoning,
and general knowledge. We just teach it medical specifics.

## Step 1: Install Dependencies (Colab)

In [ ]:
# Run this cell on Google Colab
!pip install -q transformers datasets peft bitsandbytes trl accelerate torch

In [ ]:
import torch
import os
from datetime import datetime

# Check GPU
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")
else:
    print("WARNING: No GPU detected. Training will be very slow on CPU.")
    print("Use Google Colab with a T4 GPU (free tier).")

## Step 2: Load and Prepare Dataset

In [ ]:
from datasets import load_dataset

# Load medical Q&A dataset from HuggingFace
# 47K Q&A pairs from NIH medical websites
dataset = load_dataset("keivalya/MedQuad-MedicalQnADataset", split="train")
print(f"Total samples: {len(dataset)}")
print(f"Columns: {dataset.column_names}")
print(f"\nSample:")
print(f"Q: {dataset[0]['Question'][:200]}")
print(f"A: {dataset[0]['Answer'][:200]}")

In [ ]:
# Limit dataset size (5000 samples is enough for good fine-tuning)
MAX_SAMPLES = 5000

dataset = dataset.shuffle(seed=42)
if len(dataset) > MAX_SAMPLES:
    dataset = dataset.select(range(MAX_SAMPLES))

print(f"Using {len(dataset)} samples for training")

In [ ]:
def format_for_training(example):
    """
    Format into TinyLlama chat template.
    
    The model needs to see the PATTERN it should follow:
    <|user|> question </s>
    <|assistant|> answer </s>
    """
    question = example.get("Question", "")
    answer = example.get("Answer", "")
    
    text = (
        f"<|user|>\n"
        f"You are a medical specialist. Answer this question accurately and clearly.\n\n"
        f"{question}</s>\n"
        f"<|assistant|>\n"
        f"{answer}</s>"
    )
    return {"text": text}

formatted_dataset = dataset.map(format_for_training)
split = formatted_dataset.train_test_split(test_size=0.1, seed=42)
train_data = split["train"]
val_data = split["test"]

print(f"Train: {len(train_data)} | Val: {len(val_data)}")
print(f"\nFormatted example:\n{train_data[0]['text'][:400]}...")

## Step 3: Load Base Model with 4-bit Quantization

**QLoRA = Quantized LoRA**
- Load the model in 4-bit precision (saves ~75% memory)
- Then add LoRA adapters (small trainable matrices)
- Train only the adapters (~1% of total parameters)

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

MODEL_NAME = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

# 4-bit quantization config
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",           # NormalFloat4 — best for LLMs
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,       # Nested quantization — saves more memory
)

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

# Load model in 4-bit
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
)
model.config.use_cache = False  # Required for gradient checkpointing

print(f"Model loaded: {MODEL_NAME}")
print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")
print(f"Model size in memory: ~{sum(p.numel() * p.element_size() for p in model.parameters()) / 1e6:.0f} MB")

## Step 4: Apply LoRA Adapters

LoRA adds small trainable matrices to specific layers.
We target the attention layers (q, k, v, o projections) because
that's where the model learns relationships between tokens.

In [ ]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

# Prepare model for quantized training
model = prepare_model_for_kbit_training(model)

# LoRA configuration
lora_config = LoraConfig(
    r=16,                # Rank — dimensions of the low-rank matrices
    lora_alpha=32,       # Scaling factor (usually 2x rank)
    lora_dropout=0.05,   # Dropout for regularization
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    bias="none",
    task_type="CAUSAL_LM",
)

model = get_peft_model(model, lora_config)

# Count parameters
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"Trainable parameters: {trainable:,} ({100 * trainable / total:.2f}%)")
print(f"Total parameters: {total:,}")
print(f"\nWe're only training {100 * trainable / total:.2f}% of the model!")
print(f"That's {trainable:,} out of {total:,} parameters.")

## Step 5: Train with SFTTrainer

SFTTrainer (Supervised Fine-Tuning Trainer) from the `trl` library
handles the training loop, logging, and evaluation.

In [ ]:
from transformers import TrainingArguments
from trl import SFTTrainer

OUTPUT_DIR = "./medical-specialist-qlora"

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=3,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=4,       # Effective batch size = 4 * 4 = 16
    learning_rate=2e-4,
    warmup_ratio=0.03,
    weight_decay=0.001,
    max_grad_norm=0.3,
    logging_steps=25,
    eval_strategy="steps",
    eval_steps=100,
    save_strategy="steps",
    save_steps=100,
    save_total_limit=3,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    fp16=True,                           # Mixed precision for speed
    optim="paged_adamw_32bit",           # Memory-efficient optimizer
    report_to="none",                    # No wandb/tensorboard
)

trainer = SFTTrainer(
    model=model,
    train_dataset=train_data,
    eval_dataset=val_data,
    tokenizer=tokenizer,
    args=training_args,
    max_seq_length=512,
)

print("Training setup complete. Starting training...")
print(f"Epochs: {training_args.num_train_epochs}")
print(f"Effective batch size: {training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps}")
print(f"Learning rate: {training_args.learning_rate}")

In [ ]:
# Train!
start_time = datetime.now()
train_result = trainer.train()
end_time = datetime.now()

print(f"\nTraining complete!")
print(f"Duration: {end_time - start_time}")
print(f"Final train loss: {train_result.training_loss:.4f}")

## Step 6: Plot Training Metrics

In [ ]:
import matplotlib.pyplot as plt

# Extract metrics from training history
log_history = trainer.state.log_history

train_losses = [(h["step"], h["loss"]) for h in log_history if "loss" in h and "eval_loss" not in h]
eval_losses = [(h["step"], h["eval_loss"]) for h in log_history if "eval_loss" in h]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Training loss
if train_losses:
    steps, losses = zip(*train_losses)
    axes[0].plot(steps, losses, 'b-', alpha=0.7, label='Train Loss')
    axes[0].set_xlabel('Step')
    axes[0].set_ylabel('Loss')
    axes[0].set_title('Training Loss')
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)

# Eval loss
if eval_losses:
    steps, losses = zip(*eval_losses)
    axes[1].plot(steps, losses, 'r-o', alpha=0.7, label='Eval Loss')
    axes[1].set_xlabel('Step')
    axes[1].set_ylabel('Loss')
    axes[1].set_title('Validation Loss')
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('training_metrics.png', dpi=150, bbox_inches='tight')
plt.show()
print("Training curves saved to training_metrics.png")

## Step 7: Save the LoRA Adapter

We only save the adapter weights (~10MB), not the full model (~2GB).
To use it later: load the base model + load the adapter on top.

In [ ]:
# Save adapter weights
ADAPTER_DIR = "./medical-specialist-adapter"
model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)

# Check size
adapter_size = sum(
    os.path.getsize(os.path.join(ADAPTER_DIR, f))
    for f in os.listdir(ADAPTER_DIR)
    if os.path.isfile(os.path.join(ADAPTER_DIR, f))
) / 1e6

print(f"Adapter saved to {ADAPTER_DIR}")
print(f"Adapter size: {adapter_size:.1f} MB")
print(f"\nThe base model is ~2GB. The adapter is ~{adapter_size:.0f}MB.")
print("That's the power of LoRA — you only save the small trained part.")

## Step 8: Test — Base Model vs Fine-Tuned

Let's ask the same questions to both models and compare.

In [ ]:
# Load base model (without fine-tuning) for comparison
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
)
base_tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
base_tokenizer.pad_token = base_tokenizer.eos_token

print("Base model loaded for comparison")

In [ ]:
def generate(model, tokenizer, question, max_new_tokens=256):
    prompt = (
        f"<|user|>\n"
        f"You are a medical specialist. Answer this question accurately and clearly.\n\n"
        f"{question}</s>\n"
        f"<|assistant|>\n"
    )
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=0.7,
            top_p=0.9,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id,
        )
    response = tokenizer.decode(outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    return response.strip()


test_questions = [
    "What are the common symptoms of Type 2 diabetes?",
    "How is hypertension diagnosed and what are the treatment options?",
    "What are the warning signs of a stroke?",
    "What causes iron deficiency anemia and how is it treated?",
    "What is the difference between Crohn's disease and ulcerative colitis?",
]

print("Comparing base model vs fine-tuned model...\n")
print("=" * 80)

for q in test_questions:
    print(f"\nQ: {q}")
    print("-" * 80)
    
    base_answer = generate(base_model, base_tokenizer, q)
    print(f"\nBASE MODEL:\n{base_answer[:500]}")
    
    ft_answer = generate(model, tokenizer, q)
    print(f"\nFINE-TUNED:\n{ft_answer[:500]}")
    
    print("=" * 80)

## Step 9: Download Adapter (for local use)

Download the adapter weights to use in the Streamlit app.

In [ ]:
# If on Colab, zip and download the adapter
import shutil

shutil.make_archive("medical-specialist-adapter", "zip", ADAPTER_DIR)
print("Adapter zipped. Download 'medical-specialist-adapter.zip'")
print("Extract to models/medical-specialist/ in your project directory.")

# On Colab, this will trigger a download:
try:
    from google.colab import files
    files.download("medical-specialist-adapter.zip")
except ImportError:
    print("Not on Colab — copy the adapter manually.")

## Summary

What we did:
1. Loaded 5,000 medical Q&A pairs from HuggingFace
2. Formatted them into the TinyLlama chat template
3. Loaded TinyLlama in 4-bit precision (QLoRA)
4. Added LoRA adapters (trained ~1% of parameters)
5. Fine-tuned for 3 epochs
6. Compared base vs fine-tuned responses

The fine-tuned model gives more detailed, structured, and medically accurate responses
compared to the base model — and we only trained 1% of its parameters.